In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Feedforward clasic și flux de control (circuite dinamice)

<Accordion>
<AccordionItem title="Versiuni de pachete">

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm să folosești aceste versiuni sau versiuni mai noi.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
Circuitele dinamice sunt instrumente puternice cu care poți măsura qubiți în mijlocul execuției unui Circuit cuantic și apoi efectua operații de logică clasică în cadrul circuitului, pe baza rezultatelor acelor măsurători la mijlocul circuitului. Acest proces este cunoscut și sub denumirea de _feedforward clasic_. Deși suntem la început în înțelegerea modului optim de a profita de circuitele dinamice, comunitatea de cercetare cuantică a identificat deja o serie de cazuri de utilizare, precum:

* Pregătirea eficientă a stărilor cuantice, cum ar fi [starea GHZ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), [starea W](https://arxiv.org/abs/2403.07604) (pentru mai multe informații despre starea W, consultați și ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)), și o clasă largă de [stări de produs matriceal](https://arxiv.org/abs/2404.16083)
* [Entanglement eficient pe distanțe lungi](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) între qubiți de pe același cip, folosind circuite de adâncime mică
* [Eșantionarea eficientă a circuitelor de tip IQP](https://arxiv.org/pdf/2505.04705)

Qiskit suportă patru construcții de flux de control pentru feedforward clasic, fiecare implementată ca metodă pe [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit). Construcțiile și metodele corespunzătoare sunt:

- Instrucțiunea if - [`QuantumCircuit.if_test`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test)
- Instrucțiunea switch - [`QuantumCircuit.switch`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#switch)
- Bucla for - [`QuantumCircuit.for_loop`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#for_loop)
- Bucla while - [`QuantumCircuit.while_loop`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#while_loop)

Fiecare dintre aceste metode returnează un [manager de context](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) și este folosită de obicei într-o instrucțiune `with`. Restul acestui ghid explică fiecare dintre aceste construcții și cum să le folosești.

> **Caution:** Există unele limitări ale operațiilor de feedforward clasic și flux de control pe hardware cuantic care pot afecta programul tău. Pentru mai multe informații, consultă [Execută circuite dinamice](/guides/execute-dynamic-circuits).

## Instrucțiunea `if`
Instrucțiunea `if` este folosită pentru a efectua condiționat operații pe baza valorii unui bit sau registru clasic.

În exemplul de mai jos, aplicăm o poartă Hadamard unui qubit și îl măsurăm. Dacă rezultatul este 1, atunci aplicăm o poartă X pe qubit, ceea ce are efectul de a-l readuce la starea 0. Măsurăm apoi qubit-ul din nou. Rezultatul măsurătorii ar trebui să fie 0 cu probabilitate de 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/101aaa8f-7061-4924-9b50-806d7e1ab728-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/101aaa8f-7061-4924-9b50-806d7e1ab728-0.avif)

Instrucțiunii `with` i se poate atribui o țintă care este ea însăși un manager de context, ce poate fi stocat și utilizat ulterior pentru a crea un bloc else, care se execută ori de câte ori conținutul blocului `if` *nu* este executat.

În exemplul de mai jos, inițializăm registre cu doi qubiți și două biți clasici. Aplicăm o poartă Hadamard primului qubit și îl măsurăm. Dacă rezultatul este 1, atunci aplicăm o poartă Hadamard pe al doilea qubit; altfel, aplicăm o poartă X pe al doilea qubit. În final, măsurăm și al doilea qubit.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/1f6737fe-bc45-4d0c-b7b4-1096e2d7e14a-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/1f6737fe-bc45-4d0c-b7b4-1096e2d7e14a-0.avif)

Pe lângă condiționarea pe un singur bit clasic, este posibil și să condiționezi pe valoarea unui registru clasic compus din mai mulți biți.

În exemplul de mai jos, aplicăm porți Hadamard la doi qubiți și îi măsurăm. Dacă rezultatul este `01`, adică primul qubit este 1 și al doilea qubit este 0, atunci aplicăm o poartă X unui al treilea qubit. În final, măsurăm al treilea qubit. Remarcă că, pentru claritate, am ales să specificăm starea celui de-al treilea bit clasic, care este 0, în condiția `if`. În desenul circuitului, condiția este indicată prin cercuri pe biții clasici pe care se condiționează. Un cerc plin indică condiționarea pe 1, în timp ce un cerc cu contur indică condiționarea pe 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/37ec3fa6-04b5-4165-b8d2-bae5fd238331-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/37ec3fa6-04b5-4165-b8d2-bae5fd238331-0.avif)

## Instrucțiunea switch
Instrucțiunea switch este folosită pentru a selecta acțiuni pe baza valorii unui bit sau registru clasic. Este similară cu o instrucțiune if, dar poți specifica mai multe cazuri pentru logica de ramificare. Exemplul de mai jos aplică o poartă Hadamard unui qubit și îl măsoară. Dacă rezultatul este 0, aplică o poartă X pe qubit, iar dacă rezultatul este 1, aplică o poartă Z. Rezultatul măsurătorii ar trebui să fie 1 cu probabilitate de 100%.

In [4]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.switch(c0) as case:
    with case(0):
        circuit.x(q0)
    with case(1):
        circuit.z(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/fc2bc3c3-eab1-41f0-b696-5e8b30155d55-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/fc2bc3c3-eab1-41f0-b696-5e8b30155d55-0.avif)

Deoarece exemplul de mai sus a folosit un singur bit clasic, au existat doar două cazuri posibile, deci ai fi putut obține același rezultat folosind o instrucțiune if-else. Cazul switch este util în principal când ramifici în funcție de valoarea unui registru clasic compus din mai mulți biți. Următorul exemplu arată cum să construiești un caz implicit, care se execută dacă niciunul dintre cazurile precedente nu se potrivește. Reține că într-o instrucțiune switch, se execută doar unul dintre blocuri. Nu există fallthrough.

Exemplul de mai jos aplică porți Hadamard la doi qubiți și îi măsoară. Dacă rezultatul este fie 00, fie 11, aplică o poartă Z celui de-al treilea qubit. Dacă rezultatul este 01, aplică o poartă Y. Dacă niciunul dintre cazurile precedente nu s-a potrivit, aplică o poartă X. În final, măsoară al treilea qubit.

In [5]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.switch(clbits) as case:
    with case(0b000, 0b011):
        circuit.z(q2)
    with case(0b001):
        circuit.y(q2)
    with case(case.DEFAULT):
        circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 267, '110': 249, '011': 265, '000': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/a5d43b4c-c538-4f34-8cf3-92c2c0d26fdd-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/a5d43b4c-c538-4f34-8cf3-92c2c0d26fdd-0.avif)

## Bucla for
O buclă for este folosită pentru a itera peste o secvență de valori clasice și a efectua unele operații în cadrul fiecărei iterații.

Următorul exemplu folosește o buclă for pentru a aplica 5 porți X unui qubit și apoi îl măsoară. Deoarece efectuează un număr impar de porți X, efectul general este să inverseze qubit-ul din starea 0 în starea 1.

In [6]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

with circuit.for_loop(range(5)) as _:
    circuit.x(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/53a26ce5-3564-47a0-8803-c9c46db86923-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/53a26ce5-3564-47a0-8803-c9c46db86923-0.avif)

## Bucla while
O buclă while este folosită pentru a repeta instrucțiuni cât timp o condiție este satisfăcută.

Exemplul de mai jos aplică porți Hadamard la doi qubiți și îi măsoară. Apoi creează o buclă while care repetă această procedură cât timp rezultatul măsurătorii este 11. Ca rezultat, măsurătoarea finală nu ar trebui să fie niciodată 11, celelalte posibilități apărând cu frecvență aproximativ egală.

In [7]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)

q0, q1 = qubits
c0, c1 = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.while_loop((clbits, 0b11)):
    circuit.h([q0, q1])
    circuit.measure(q0, c0)
    circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 334, '10': 368, '00': 322}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/174a9675-3c8b-4b5e-808e-f7e0f8b9c805-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/174a9675-3c8b-4b5e-808e-f7e0f8b9c805-0.avif)

## Expresii clasice
Modulul de expresii clasice Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) conține o reprezentare exploratorie a operațiilor de execuție pe valori clasice în timpul execuției circuitului.

Următorul exemplu arată că poți folosi calculul parității pentru a crea o stare GHZ cu n qubiți folosind circuite dinamice. Mai întâi, generează $n/2$ perechi Bell pe qubiți adiacenți. Apoi, lipește aceste perechi împreună folosind un strat de porți CNOT între perechi. Măsori apoi qubit-ul țintă al tuturor porților CNOT anterioare și resetezi fiecare qubit măsurat la starea $\vert 0 \rangle$. Aplici $X$ la fiecare poziție nemăsurată pentru care paritatea tuturor biților precedenți este impară. În final, porțile CNOT sunt aplicate qubiților măsurați pentru a restabili entanglement-ul pierdut în măsurătoare.

În calculul parității, primul element al expresiei construite implică ridicarea obiectului Python `mr[0]` la un nod [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` este folosit pentru a transforma obiecte arbitrare în expresii clasice). Acest lucru nu este necesar pentru `mr[1]` și posibilul registru clasic următor, deoarece acestea sunt intrări pentru `expr.bit_xor`, iar orice ridicare necesară este efectuată automat în aceste cazuri. Astfel de expresii pot fi construite în bucle și alte constructe.

In [8]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [9]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d2fdf38a-e874-4de1-9a79-08aab97f9ecc-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d2fdf38a-e874-4de1-9a79-08aab97f9ecc-0.avif)

<span id="store"></span>
### `Store`

Poți utiliza instrucțiunea [`store`](https://docs.quantum.ibm.com/api/qiskit/circuit#store) pentru a salva rezultatul unei expresii clasice, dacă acea expresie va fi utilizată în mod repetat. Operațiile sunt paralelizate automat, făcând codul tău semnificativ mai eficient la rulare.

De exemplu, este mai natural și mai eficient la rulare să scrii $B[0] \oplus B[1] \oplus B[2] \ldots$, unde $B = \neg A$, decât $(\neg A[0]) \oplus (\neg A[1]) \oplus (\neg A[2]) \ldots$.  Prima formă calculează negarea într-un singur pas paralel înainte de lanțul XOR, în loc să evalueze fiecare negare secvențial în interiorul expresiei.

Exemplu complet:

In [10]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

qregs = QuantumRegister(4, "q")
creg = ClassicalRegister(3, "c")
# temp is a plain ClassicalRegister used as the store target
temp = ClassicalRegister(3, "temp")
qc = QuantumCircuit(qregs, creg, temp)

qc.h([0, 1, 2])
qc.measure([0, 1, 2], creg)

# Store bit-NOT of the full 3-bit register into temp
qc.store(temp, expr.bit_not(creg))

# Compute parity of temp using bit-indexed XOR
parity = expr.bit_xor(
    expr.bit_xor(expr.index(temp, 0), expr.index(temp, 1)),
    expr.index(temp, 2),
)

# Flip q3 if parity of ~creg is 1
with qc.if_test(parity):
    qc.x(3)

qc.measure([0, 1, 2], creg)

qc.draw("mpl")

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/f76db731-afa1-4777-9482-25376aa86175-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/f76db731-afa1-4777-9482-25376aa86175-0.avif)

## Pași următori
> **Tip:** - Învață cum să implementezi un decuplaj dinamic precis folosind [stretch](/guides/stretch).
> - Folosește [vizualizarea programului circuitului](/guides/qiskit-runtime-circuit-timing) pentru a depana și optimiza circuitele tale dinamice.
> - [Execută circuite dinamice](/guides/execute-dynamic-circuits).